# Python challenges 9: backtracking, heaps, intervals and design

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/practice/python-challenges-09-backtracking-heaps-intervals-design.ipynb)

Exhaustive search with backtracking, a heap for the k-th item, sorting intervals, and a design question that combines a hash map with a linked list. Statements are paraphrased, and each solution is collapsed under its exercise.

Try each exercise in the code cell below its question. Use the sample inputs to compare your result with the expected output. Expand **Show solution** when you want to review the approach.

## Exercise 1: Subsets

**Difficulty:** Medium · about 15 min

Return every subset of the distinct values in `nums`. The order of the subsets does not matter.

In a pipeline: SQL's `GROUP BY CUBE` computes an aggregate for every subset of the grouping columns, which is exactly this enumeration.

**Example 1**

```text
Input:  nums = [1, 2, 3]
Output: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
```

**Example 2**

```text
Input:  nums = [0]
Output: [[], [0]]
```

<details>
<summary>💡 Tip</summary>

For each number, a subset either includes it or skips it. Build the choices one value at a time: record a copy of the current path at every step, then extend it with each later value and backtrack.

</details>

In [ ]:
def subsets(nums):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
def subsets(nums):
    # Approach: backtracking. Every subset is a path in a decision tree: at each value,
    # include it or not. Recording the current list at every node gives all subsets,
    # because each prefix of a choice sequence is itself a subset.
    # Copy the list when recording (current[:]), since current keeps changing.
    # Complexity: O(n * 2^n) time, O(n) recursion depth.
    result = []

    def build(start, current):
        result.append(current[:])            # record the subset chosen so far
        for i in range(start, len(nums)):    # only later values, so no subset repeats in a new order
            current.append(nums[i])
            build(i + 1, current)
            current.pop()                    # undo the choice before trying the next value

    build(0, [])
    return result
```

</details>

## Exercise 2: Permutations

**Difficulty:** Medium · about 15 min

Return every ordering of the distinct values in `nums`.

**Example 1**

```text
Input:  nums = [1, 2, 3]
Output: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]
```

**Example 2**

```text
Input:  nums = [1]
Output: [[1]]
```

<details>
<summary>💡 Tip</summary>

Fill the path one position at a time. At each position, try every value not yet used. After the recursive call, remove the value again, which is the backtrack step.

</details>

In [ ]:
def permutations(nums):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
def permutations(nums):
    # Approach: backtracking by choice. At each level pick one remaining value to place next.
    # Each path through the tree uses every value once, which gives every ordering.
    # The "remaining" list is rebuilt without the picked value, so nothing is reused by mistake.
    # Complexity: O(n * n!) time, O(n) recursion depth plus the output.
    result = []

    def build(current, remaining):
        if not remaining:                    # every value placed: one complete permutation
            result.append(current[:])
            return
        for i, value in enumerate(remaining):
            build(current + [value], remaining[:i] + remaining[i + 1:])

    build([], list(nums))
    return result
```

</details>

## Exercise 3: Combination Sum

**Difficulty:** Medium · about 20 min

Given distinct positive `candidates`, return every combination that sums to `target`. Each candidate may be used any number of times. The order of combinations does not matter.

**Example 1**

```text
Input:  candidates = [2, 3, 6, 7], target = 7
Output: [[2, 2, 3], [7]]
```

**Example 2**

```text
Input:  candidates = [2], target = 1
Output: []
```

<details>
<summary>💡 Tip</summary>

Sort the candidates so you can stop early. Subtract each candidate from the remaining target. Because a candidate may repeat, the next call starts at the same index, not the next one. Stop when the remainder is zero (record it) or negative (give up).

</details>

In [ ]:
def combination_sum(candidates, target):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
def combination_sum(candidates, target):
    # Approach: backtracking with a remaining sum. Only candidates that fit are tried.
    # Pass the index i (not i + 1) on recursion, because a candidate may be reused.
    # Starting each loop at "start" avoids producing the same combination in a different order.
    # Complexity: exponential in target / min(candidates) in the worst case.
    result = []

    def build(start, current, remaining):
        if remaining == 0:                   # exact hit: record the combination
            result.append(current[:])
            return
        for i in range(start, len(candidates)):
            if candidates[i] <= remaining:   # prune: skip values that overshoot
                current.append(candidates[i])
                build(i, current, remaining - candidates[i])   # i, not i + 1: reuse allowed
                current.pop()

    build(0, [], target)
    return result
```

</details>

## Exercise 4: Kth Largest Element in a Stream

**Difficulty:** Easy · about 15 min

Write a class `KthLargest` with `__init__(k, nums)` and `add(value)`. `add` stores a new value and returns the `k`-th largest value seen so far.

In a pipeline: a size-k min-heap is how you keep the top N rows per key (top products per store, slowest queries per day) in one pass without sorting everything.

**Example 1**

```text
Input:  k = 3, nums = [4, 5, 8, 2]; add(3), add(5), add(10), add(9), add(4)
Output: [4, 5, 5, 8, 8]
```

<details>
<summary>💡 Tip</summary>

Keep a min-heap of at most k values. Its smallest item is the kth largest value seen so far. After each `heappush`, pop the smallest if the heap has grown past k.

</details>

In [ ]:
import heapq

class KthLargest:
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
import heapq

class KthLargest:
    # Approach: a min-heap that holds only the k largest values seen so far.
    # Its smallest item (the root) is the k-th largest. Anything smaller than the root
    # can never affect the answer, so it is dropped as soon as the heap is over size k.
    # Complexity: O(log k) per add, O(k) space.
    def __init__(self, k, nums):
        self._k = k
        self._heap = []
        for value in nums:
            self.add(value)

    def add(self, value):
        heapq.heappush(self._heap, value)
        if len(self._heap) > self._k:
            heapq.heappop(self._heap)     # remove the smallest, which is outside the top k
        return self._heap[0]
```

</details>

## Exercise 5: Merge Intervals

**Difficulty:** Medium · about 15 min

`intervals` is a list of `[start, end]` pairs. Merge every pair of overlapping or touching intervals. Return the merged list, sorted by start.

In a pipeline: the same sweep merges overlapping time ranges or partition windows, for example to work out which date ranges a backfill must actually cover.

**Example 1**

```text
Input:  intervals = [[1, 3], [2, 6], [8, 10], [15, 18]]
Output: [[1, 6], [8, 10], [15, 18]]
```

**Example 2**

```text
Input:  intervals = [[1, 4], [4, 5]]
Output: [[1, 5]]
```

<details>
<summary>💡 Tip</summary>

Sort by start. Keep the last merged interval. If the next interval starts at or before its end, extend the end to the larger of the two ends. Otherwise start a new interval. Touching intervals, such as [1, 4] and [4, 5], also merge.

</details>

In [ ]:
def merge_intervals(intervals):
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
def merge_intervals(intervals):
    # Approach: sort by start. Then an interval overlaps the last merged one exactly when
    # its start is at or before the last merged end. Overlaps extend the end; gaps start a new group.
    # Complexity: O(n log n) time for the sort, O(n) space for the result.
    merged = []
    for start, end in sorted(intervals):    # sorting makes overlaps neighbours in the list
        if merged and start <= merged[-1][1]:   # touching (start == end) also merges
            merged[-1][1] = max(merged[-1][1], end)   # max: one interval may sit inside another
        else:
            merged.append([start, end])
    return merged
```

</details>

## Exercise 6: LRU Cache

**Difficulty:** Medium · about 25 min

Write `LRUCache(capacity)` with two methods:

- `get(key)` returns the value, or `-1` when the key is absent. Reading a key marks it as recently used.
- `put(key, value)` stores a value. When the cache is full, evict the least recently used key first.

In a pipeline: an LRU cache is the usual way to bound an in-memory lookup cache when enriching a stream against a slow dimension table or API.

**Example 1**

```text
Input:  capacity = 2; put(1, 1), put(2, 2), get(1), put(3, 3), get(2), put(4, 4), get(1), get(3), get(4)
Output: [None, None, 1, None, -1, None, -1, 3, 4]
```

**Example 2**

```text
Input:  capacity = 1; put(2, 1), get(2), put(3, 2), get(2), get(3)
Output: [None, 1, None, -1, 2]
```

<details>
<summary>💡 Tip</summary>

Use an `OrderedDict`, keeping the least recently used key first. `move_to_end(key)` marks a key as recent on each get and put. When the cache is over capacity, `popitem(last=False)` removes the oldest key.

</details>

In [ ]:
from collections import OrderedDict

class LRUCache:
    # your code here
    pass

<details>
<summary><b>Show solution</b></summary>

```python
from collections import OrderedDict

class LRUCache:
    # Approach: OrderedDict keeps keys in access order. The most recently used key is moved
    # to the end on every get or put. The front key is always the least recently used one.
    # Complexity: O(1) average for get and put, O(capacity) space.
    def __init__(self, capacity):
        self._capacity = capacity
        self._data = OrderedDict()

    def get(self, key):
        if key not in self._data:
            return -1
        self._data.move_to_end(key)       # reading counts as a use
        return self._data[key]

    def put(self, key, value):
        if key in self._data:
            self._data.move_to_end(key)   # an update is also a use
        self._data[key] = value
        if len(self._data) > self._capacity:
            self._data.popitem(last=False)   # evict the least recently used (the front)
```

</details>